In [1]:
import os
from pathlib import Path

## where are traces data
trace_root = "traces_data"
os.makedirs(trace_root, exist_ok=True)

# import project utilities
from langfuse_utils import *


## Setup for Langfuse API and fetching Traces

In [1]:
## Load required keys from .env
from dotenv import load_dotenv
_ = load_dotenv('../.env')

SECRET_KEY = os.environ.get("LANGFUSE_SECRET_KEY")
PUBLIC_KEY = os.environ.get("LANGFUSE_PUBLIC_KEY")
BASE_URL = os.environ.get("LANGFUSE_BASE_URL")


In [2]:
# imports for working with traces in Langfuse
import requests

# Authentication
import base64
auth = base64.b64encode(f"{PUBLIC_KEY}:{SECRET_KEY}".encode()).decode()
H = {"Authorization": f"Basic {auth}"}

In [3]:
## Filters for Langfuse
# NAME_FILTER = "LangGraph"     # or None


In [4]:
# imports for notebook features
# import ipywidgets as w

In [5]:

start = datetime.fromisoformat("2025-07-25T14:30:00+00:00")
end   = datetime.fromisoformat("2025-07-25T15:30:00+00:00")

params = {
    "fromTimestamp": iso_utc_plus(start),
    "toTimestamp": iso_utc_plus(end),
    "limit": LIMIT,
}
r = requests.get(f"{BASE_URL}/api/public/traces", headers=H, params=params, timeout=20)
print(r.status_code, r.text[:400])


200 {"data":[{"id":"0a3d6a938d216d42d8282ba1f55a7c64","projectId":"wri_lcl","name":"LangGraph","timestamp":"2025-07-25T15:24:34.262Z","environment":"default","tags":[],"bookmarked":false,"release":null,"version":null,"userId":null,"sessionId":null,"public":false,"input":{"messages":[{"content":"","additional_kwargs":{},"response_metadata":{},"type":"human","name":null,"id":null,"example":false},{"cont


## Time slots to fetch

In [ ]:
# Provide time slots in CENTRAL TIME (CT) 
slots = [
    ("Ana Benavides",         "2025-07-25 09:30"),
    ("Kuang Keng Kuek Ser",   "2025-07-29 07:00"),
    ("Willie",                "2025-07-29 16:00"),
    ("Berton Pakpahan",       "2025-08-01 09:00"),
]

# Set this appropriately
INTERVIEW_LENGTH_IN_MINUTES = 60

In [ ]:
for who, ct_start in slots:
    f_iso, t_iso = ct_window_iso(ct_start, INTERVIEW_LENGTH_IN_MINUTES)
    rows = fetch_window(f_iso, t_iso, BASE_URL, H)
    
    # client-side filter to avoid server 400s on unknown params
    # if NAME_FILTER: rows = [r for r in rows if r.get("name") == NAME_FILTER]

    slug = "".join(c.lower() if c.isalnum() else "_" for c in who)
    folder = os.path.join(trace_root, slug)

    save_jsonl(os.path.join(folder, "traces_list.jsonl"), rows)
    save_csv(os.path.join(folder, "traces_summary.csv"), [summarize(r) for r in rows])

    print(f"{who}: {ct_start} CT → "
      f"{datetime.fromisoformat(f_iso).strftime('%H:%M')} .. "
      f"{datetime.fromisoformat(t_iso).strftime('%H:%M')} UTC | {len(rows)} traces")

## List all saved traces

In [3]:
!tree traces_data/

traces_data/
├── ana_benavides__sgf_
│   ├── traces_list.jsonl
│   └── traces_summary.csv
├── berton_pakpahan__ifmn_
│   ├── traces_list.jsonl
│   └── traces_summary.csv
├── kuang_keng_kuek_ser__pulitzer_
│   ├── traces_list.jsonl
│   └── traces_summary.csv
└── willie__mongabay_
    ├── traces_list.jsonl
    └── traces_summary.csv

5 directories, 8 files


## Load saved traces in to dataframes

In [2]:
ROOT = Path("traces_data")
assert ROOT.exists(), f"Folder not found: {ROOT.resolve()}"

In [3]:
# Load all CSVs with interview label
trace_rows = []
jsonl_index = {}  # interview -> {trace_id: full_row}
for traces_dir in sorted(d for d in ROOT.iterdir() if d.is_dir()):
    label = traces_dir.name
    csv_path   = traces_dir / "traces_summary.csv"
    jsonl_path = traces_dir / "traces_list.jsonl"

    idf = safe_read_csv(csv_path)
    if idf.empty:
        continue
    idf["interview"] = label

    # Timestamps in CT for readability
    if "timestamp" in idf:
        idf["timestamp_ct"] = idf["timestamp"].dt.tz_convert(ZoneInfo("America/Chicago"))
        idf["date_ct"] = idf["timestamp_ct"].dt.date
        idf["minute_ct"] = idf["timestamp_ct"].dt.floor("min")
    trace_rows.append(idf)

    # Minimal JSONL index for drilldowns
    idx = {}
    for rec in read_jsonl(jsonl_path):
        tid = rec.get("id")
        if tid:
            idx[tid] = rec
    jsonl_index[label] = idx

df = pd.concat(trace_rows, ignore_index=True)

In [4]:

# derive some new columns for convenience 

df["timestamp_ct"] = pd.to_datetime(df["timestamp_ct"], errors="coerce")
df["duration_s"] = pd.to_numeric(df["duration_s"], errors="coerce")
df["totalCost"] = pd.to_numeric(df["totalCost"], errors="coerce")
cats = pd.CategoricalDtype(["SUCCESS","PARTIAL","ERROR","NO_TOOLS"], ordered=True)
df["outcome_status"] = df["outcome_status"].astype(cats)

# Convenience booleans
df["used_pick_aoi"] = df["tool_names"].fillna("").str.contains(r"\bpick-aoi\b")
df["used_pick_dataset"] = df["tool_names"].fillna("").str.contains(r"\bpick-dataset\b")
df["used_pull_data"] = df["tool_names"].fillna("").str.contains(r"\bpull-data\b")
df["has_aoi"] = df["selected_aoi"].notna() & (df["selected_aoi"].astype(str).str.len()>0)
df["has_dataset"] = df["selected_dataset"].notna() & (df["selected_dataset"].astype(str).str.len()>0)
df["apology_like"] = df["ai_output_trunc"].str.contains(r"apolog|unable to|failed to|cannot|can\'t|can't", case=False, na=False)


In [5]:
df.head(2).T

,0,1
ai_output_trunc,"I apologize, but I'm unable to pull the distur...",NaN
createdAt_utc,2025-07-25T15:25:12Z,2025-07-25T15:23:16Z
dataset_context_layer,"Classified drivers of DIST Alerts (aka LDACS),...",NaN
duration_bucket,>10s,0.5–1s
duration_s,34.911,0.72
end_date,2024-12-31,NaN
environment,default,default
final_ai_tokens_out,128,NaN
has_output,True,False
has_tool_failure,True,False


## Scoreboard! 

In [6]:
SHOW_COVERAGE = False   # set to False to hide the "Coverage" section in the display

## Create scoreboard
scoreboard = (
    df.groupby("interview")
      .agg(
          traces=("id","count"),
          success=("outcome_status", lambda s: (s=="SUCCESS").sum()),
          partial=("outcome_status", lambda s: (s=="PARTIAL").sum()),
          error=("outcome_status", lambda s: (s=="ERROR").sum()),
          no_tools=("outcome_status", lambda s: (s=="NO_TOOLS").sum()),
          aoi_cov=("has_aoi","sum"),
          dataset_cov=("has_dataset","sum"),
          pick_aoi=("used_pick_aoi","sum"),
          pick_dataset=("used_pick_dataset","sum"),
          pull_data=("used_pull_data","sum"),
          med_dur_s=("duration_s","median"),
          p95_dur_s=("duration_s", p95),
          cost_total=("totalCost","sum"),
          cost_median=("totalCost","median"),
      )
      .sort_values(["error","partial","success"], ascending=[False,False,True])
)


# Compute rates or metrics
scoreboard["success_rate"] = (scoreboard["success"] / scoreboard["traces"]).round(3)
scoreboard["coverage_aoi_rate"] = (scoreboard["aoi_cov"] / scoreboard["traces"]).round(3)
scoreboard["coverage_dataset_rate"] = (scoreboard["dataset_cov"] / scoreboard["traces"]).round(3)


In [7]:

# ---- RENAME COLUMNS ------------------------------------------------------
rename_map = {
    "traces": "traces_total",
    "success": "ntraces_success",
    "partial": "ntraces_partial",
    "error": "ntraces_error",
    "no_tools": "ntraces_no_tools",

    "aoi_cov": "aoi_covered_count",
    "dataset_cov": "dataset_covered_count",
    "coverage_aoi_rate": "aoi_coverage_rate",
    "coverage_dataset_rate": "dataset_coverage_rate",

    "pick_aoi": "tool_pick_aoi_count",
    "pick_dataset": "tool_pick_dataset_count",
    "pull_data": "tool_pull_data_count",

    "med_dur_s": "latency_median_sec",
    "p95_dur_s": "latency_p95_sec",

    "cost_median": "cost_median_usd",
    "cost_total": "cost_total_usd",
}
scoreboard = scoreboard.rename(columns=rename_map)

# ---- SECTION ORDER / SELECTION ------------------------------------------
sections = {
    "Volume": ["traces_total"],
    "Outcomes": ["ntraces_success","ntraces_partial","ntraces_error","ntraces_no_tools","success_rate"],
    "Coverage": ["aoi_covered_count","dataset_covered_count","aoi_coverage_rate","dataset_coverage_rate"],
    "Tool usage": ["tool_pick_aoi_count","tool_pick_dataset_count","tool_pull_data_count"],
    "Latency": ["latency_median_sec","latency_p95_sec"],
    "Cost (USD)": ["cost_median_usd","cost_total_usd"],
}

# drop Coverage section from the display if requested
if not SHOW_COVERAGE:
    sections.pop("Coverage", None)

# (keep only columns that exist)
ordered_metrics = [m for sec in sections.values() for m in sec if m in scoreboard.columns]
sb = scoreboard.reindex(columns=ordered_metrics)


In [8]:
sb_T = sb.T
# Build a MultiIndex on rows: (Section, Metric)
row_to_section = {m: sec for sec, metrics in sections.items() for m in metrics}
sb_T.index = pd.MultiIndex.from_tuples([(row_to_section.get(r, "Other"), r) for r in sb_T.index],
                                       names=["section","metric"])

# ---- FORMATTING HELPERS --------------------------------------------------
def fmt_secs(x):
    if pd.isna(x): return ""
    x = float(x)
    if x < 90: return f"{x:,.0f}s"
    m, s = divmod(int(round(x)), 60)
    return f"{m}m{s:02d}s"

idx = pd.IndexSlice

# error-share per interview (columns in sb_T)
err_share = (scoreboard.get("ntraces_error", 0) / scoreboard.get("traces_total", 1)).reindex(sb.index).fillna(0.0)

def highlight_error_cols(col: pd.Series):
    r = float(err_share.get(col.name, 0.0))
    if r >= 0.30:  color = "#ffd6d6"
    elif r >= 0.15: color = "#ffefef"
    else:          color = ""
    return [f"background-color: {color}" if color else "" for _ in col]



In [9]:
# ---- STYLER --------------------------------------------------------------
styler_T = (
    sb_T.style
      # formats by row groups
      .format("{:.1%}".format, subset=idx[("Outcomes","success_rate"), :])
      .format(fmt_secs, subset=idx[("Latency", ["latency_median_sec","latency_p95_sec"]), :])
      .format(lambda v: "" if pd.isna(v) else f"${v:,.2f}", subset=idx[("Cost (USD)", ["cost_median_usd","cost_total_usd"]), :])

      # bars for rates (row-wise) — only outcomes success_rate, and coverage rates if shown
      .bar(subset=idx[("Outcomes","success_rate"), :], align="mid", vmin=0, vmax=1, color=None)
)

# coverage rates bars only if displayed
if SHOW_COVERAGE and {"aoi_coverage_rate","dataset_coverage_rate"} <= set(sb.columns) | set(sb.columns):
    styler_T = styler_T.bar(
        subset=idx[("Coverage", ["aoi_coverage_rate","dataset_coverage_rate"]), :],
        vmin=0, vmax=1, color=None
    )

# gentle gradient for latency & cost (legible text; no bar overlays)
styler_T = (
    styler_T
      #.background_gradient(subset=idx[("Latency", ["latency_median_sec","latency_p95_sec"]), :], cmap="Greys")
      #.background_gradient(subset=idx[("Cost (USD)", ["cost_median_usd","cost_total_usd"]), :], cmap="Greys")
      # column-wise banding by error share
      .apply(highlight_error_cols, axis=0)
      .set_caption("Traces Scoreboard")
      .set_properties(subset=idx[:, :], **{"white-space": "nowrap"})
)

In [10]:
styler_T  # display

# Tool

In [14]:

# 3) Tool reliability
def explode_tools(df):
    tmp = df.copy()
    tmp["tool_names_list"] = tmp["tool_names"].fillna("").apply(lambda s: [t for t in s.split("|") if t])
    return tmp.explode("tool_names_list")
tools_long = explode_tools(df)
tool_reliability = (tools_long.groupby("tool_names_list")
    .agg(used_in_traces=("id","count"),
         error_traces=("outcome_status", lambda s: (s=="ERROR").sum()),
         success_traces=("outcome_status", lambda s: (s=="SUCCESS").sum()),
         med_dur_s=("duration_s","median"),
         p95_dur_s=("duration_s", p95))
    .sort_values("used_in_traces", ascending=False)
)
tool_reliability["error_rate"] = (tool_reliability["error_traces"]/tool_reliability["used_in_traces"]).round(3)


In [15]:
tool_reliability

,used_in_traces,error_traces,success_traces,med_dur_s,p95_dur_s,error_rate
tool_names_list,,,,,,
pick-aoi,34,15,17,39.7115,127.61715,0.441
pick-dataset,34,15,17,39.7115,127.61715,0.441
pull-data,32,15,17,38.6025,131.84405,0.469
generate_insights,22,8,14,39.7115,152.15115,0.364


## Scrap Analysis

In [11]:

# 2) Funnel per interview
funnel = (df.groupby("interview")
    .agg(
        step0_traces=("id","count"),
        step1_pick_aoi=("used_pick_aoi","sum"),
        step2_pick_dataset=("used_pick_dataset","sum"),
        step3_pull_data=("used_pull_data","sum"),
        step4_extracted=("id", lambda s: int(((df.loc[s.index,"has_aoi"]) & (df.loc[s.index,"has_dataset"])).sum())),
        step5_success=("outcome_status", lambda s: (s=="SUCCESS").sum()),
    )
)
for i in range(1,6):
    funnel[f"step{i}_rate"] = (funnel[f"step{i}_pick_aoi" if i==1 else
                                 f"step{i}_pick_dataset" if i==2 else
                                 f"step{i}_pull_data" if i==3 else
                                 f"step{i}_extracted" if i==4 else
                                 f"step{i}_success"] / funnel["step0_traces"]).round(3)


In [13]:
funnel.T

interview,ana_benavides__sgf_,kuang_keng_kuek_ser__pulitzer_,willie__mongabay_
step0_traces,10.0,15.000,28.000
step1_pick_aoi,5.0,6.000,23.000
step2_pick_dataset,5.0,6.000,23.000
step3_pull_data,5.0,5.000,22.000
step4_extracted,5.0,6.000,23.000
step5_success,4.0,2.000,11.000
step1_rate,0.5,0.400,0.821
step2_rate,0.5,0.400,0.821
step3_rate,0.5,0.333,0.786
step4_rate,0.5,0.400,0.821
